# LangChain: RAG & Q&A

## Outline
* RAG concept
* Build a knowledge base (loader → split → embed → store)
* 2-Step RAG with LCEL
* Agentic RAG with `create_agent`
* Comparison of the two approaches


In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

_ = load_dotenv(find_dotenv())

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0, api_key=api_key, base_url=base_url)
ollama = init_chat_model("llama3.1:8b ", model_provider="ollama", temperature=0)

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


C:\Users\meisa\AppData\Local\Temp\ipykernel_67832\3174759150.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
W0819 14:43:33.397000 67832 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0819 14:43:33.661000 67832 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


## 1. Load PDF



In [10]:
# ── Step 1: Load (modified for PDF) ──

#pip install pypdf
from langchain_community.document_loaders import PyPDFLoader

# Load the PDF file
#loader = PyPDFLoader("sample_doc.pdf")
loader = PyPDFLoader("Meisam-Ghasemi-CV.pdf")
docs = loader.load()

print(f"Loaded {len(docs)} pages from PDF")
# Display part of the first page for testing
print(f"Sample content: {docs[0].page_content[:100]}...")

Loaded 2 pages from PDF
Sample content: Meisam Ghasemi Bostanabad
High Energy Physicist | Machine Learning Researcher
mghasemi@ipm.ir | www....


In [11]:
docs

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-07T15:01:05+00:00', 'author': '', 'keywords': '', 'moddate': '2026-08-07T15:01:05+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Meisam-Ghasemi-CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Meisam Ghasemi Bostanabad\nHigh Energy Physicist | Machine Learning Researcher\nmghasemi@ipm.ir | www.meisamghasemi.com | github.com/mghasemi19\nSummary\nI am an experimental particle physicist with experience in the ATLAS and CMS collaborations within the Particle Physics Group at IPM.\nMy research focuses on searches for physics beyond the Standard Model, with an emphasis on Electroweak and Strong Supersymmetry\nin hadronic ﬁnal states. I also work extensively on classical and quantum machine learning applications for accelerators and de

## 2. Build a Knowledge Base

In [14]:
# ── Step 2: Split ──
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)
splits = splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks")


Split into 11 chunks


In [15]:
splits

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-07T15:01:05+00:00', 'author': '', 'keywords': '', 'moddate': '2026-08-07T15:01:05+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Meisam-Ghasemi-CV.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='Meisam Ghasemi Bostanabad\nHigh Energy Physicist | Machine Learning Researcher\nmghasemi@ipm.ir | www.meisamghasemi.com | github.com/mghasemi19\nSummary\nI am an experimental particle physicist with experience in the ATLAS and CMS collaborations within the Particle Physics Group at IPM.\nMy research focuses on searches for physics beyond the Standard Model, with an emphasis on Electroweak and Strong Supersymmetry\nin hadronic ﬁnal states. I also work extensively on classical and quantum machine learning applications for accelerators and de

In [16]:
# ── Step 3: Embed & Store ──
embeddings = OpenAIEmbeddings(model="text-embedding-3-large", api_key=api_key, base_url=base_url)

In [ ]:
# Build the vector store
vectorstore = FAISS.from_documents(splits, embeddings)
print("Vector store created!")

# Test similarity search
query = "When are applicants registered?"
#query = "Whose this CV is for?"
results = vectorstore.similarity_search(query, k=2)
print(f"\nTop {len(results)} results for '{query}':")
for i, doc in enumerate(results):
    print(f"  {i+1}. {doc.page_content[:100]}")


Vector store created!

Top 2 results for 'Whose this CV is for?':
  1. external MCP tools, and enterprise knowledge sources to provide grounded responses ὑ7GitHub link.
Un
  2. versioned, and monitored using MLﬂow. ὑ7GitHub link, ὑ7paper.
• Developing an educational NLP tutori


In [ ]:
#query = "Who is this CV for?"
results = vectorstore.similarity_search(query, k=2)
print(f"\nTop {len(results)} results for '{query}':")
for i, doc in enumerate(results):
    print(f"  {i+1}. {doc.page_content[:100]}")


Top 2 results for 'Who is this CV for?':
  1. external MCP tools, and enterprise knowledge sources to provide grounded responses ὑ7GitHub link.
Un
  2. Meisam Ghasemi Bostanabad
High Energy Physicist | Machine Learning Researcher
mghasemi@ipm.ir | www.


In [26]:
import re

# Function to remove extra line breaks and clean the text
def clean_text(text):
    # Remove extra line breaks (\n) and replace them with spaces
    text = re.sub(r'\n+', ' ', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

loader = PyPDFLoader("sample_doc.pdf")
#loader = PyPDFLoader("Meisam-Ghasemi-CV.pdf")
docs = loader.load()

for doc in docs:
    doc.page_content = clean_text(doc.page_content)

# The texts are now clean and you can continue with the next step (Splitter)
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, # Use a slightly larger chunk size so the text remains more coherent
    chunk_overlap=100,
)
splits = splitter.split_documents(docs)

Ignoring wrong pointing object 105 0 (offset 0)


In [28]:
len(splits)

16

In [31]:
# Build the vector store
vectorstore = FAISS.from_documents(splits, embeddings)
print("Vector store created!")

results = vectorstore.similarity_search(query, k=2)
print(f"\nTop {len(results)} results for '{query}':")
for i, doc in enumerate(results):
    print(f"  {i+1}. {doc.page_content}")
    print("-"*10)


Vector store created!

Top 2 results for 'When are applicants registered?':
  1. ﮔﯾری، ﺑﮫ طور اﺳﺗﺎﻧدارد و ﮐﯾﻔﯽ ﺗوﺳط وزارت آﻣوزش و ﭘرورش ﺑﮫ ﺻورت ﺳراﺳری و ﻧﮭﺎﯾﯽ)ﮐﺗﺑﯽ( در ﺳﻧوات ﻣﺧﺗﻠف ﻣطﺎﺑق ﻣﺻوﺑﺎت ﺷورای ﻋﺎﻟﯽ آﻣوزش و ﭘرورش ﺑرﮔزار ﺷده ﺑﺎﺷد. دوره :ﻣﺗوﺳطﮫ اﯾن دوره ﺷﺎﻣل ﻧظﺎم ھﺎی آﻣوزﺷﯽ دوره ﺷش ﺳﺎﻟﮫ ﻣﺗوﺳطﮫ، دوره ﭼﮭﺎرﺳﺎﻟﮫ ﻣﺗوﺳطﮫ، دوره ﻣﺗوﺳطﮫ ﺳﮫ ﺳﺎﻟﮫ ﻧﯾم ﺳﺎﻟﯽ واﺣدی/ ﺳﺎﻟﯽ واﺣدی، دوره ﭘﯾش داﻧﺷﮕﺎھﯽ ﻧﯾم ﺳﺎﻟﯽ واﺣدی/ ﺳﺎﻟﯽ واﺣدی و دوره دوم ﻣﺗوﺳطﮫ ﻧظﺎم 3 - 3 - 3 - 3 ﻣﯽ ﺑﺎﺷد اﯾﺟﺎد ﺳﺎﺑﻘﮫ :ﺗﺣﺻﯾﻠﯽ ﺷرﮐت اﻓراد واﺟد ﺷراﯾط و ﻓﺎﻗد ﺳﺎﺑﻘﮫ ﺗﺣﺻﯾﻠﯽ در آزﻣون ھﺎی ﻧﮭﺎﯾﯽ ﻧوﺑت ﺧردادﻣﺎه. ﺗرﻣﯾم ﺳﺎﺑﻘﮫ :ﺗﺣﺻﯾﻠﯽ ﺷرﮐت ﻣﺟدد اﻓراد واﺟد ﺷراﯾط و دارای ﺳﺎﺑﻘﮫ ﺗﺣﺻﯾﻠﯽ در ﻣﺟﻣوﻋﮫ آزﻣون دروس ﻧﮭﺎﯾﯽ)دارای ﺳﺎﺑﻘﮫ( ﯾﮏ ﭘﺎﯾﮫ. دروس ﻣؤﺛر: دروس ﻧﮭﺎﯾﯽ ﮐﮫ ﺑر اﺳﺎس ﻧﺗﺎﯾﺞ آﻧﮭﺎ ﻧﻣره ﮐل ﺳﺎﺑﻘﮫ ﺗﺣﺻﯾﻠﯽ ﻣﺣﺎﺳﺑﮫ ﻣﯽ ﺷود. ﻧﻣره ﮐل ﺳﺎﺑﻘﮫ :ﺗﺣﺻﯾﻠﯽ ﺑراﺳﺎس ﺑﻧد 2 - 2 ﻣﺻوﺑﮫ ﺟﻠﺳﮫ 843 ﻣورخ 15/04/1400 ﺷورای ﻋﺎﻟﯽ اﻧﻘﻼب ﻓرھﻧﮕﯽ ﻧﻣره ﮐل ﺳﺎﺑﻘﮫ ﺗﺣﺻﯾﻠﯽ ﻋﺑﺎرت اﺳت از ﻣﯾﺎﻧﮕﯾن ﻧﻣره وزﻧﯽ ﻧﻣرات ﺗراز ﺷده دروس ﻋﻣوﻣﯽ و ﺗﺧﺻﺻﯽ. ھﺎی ﮔروه آزﻣﺎﯾﺷﯽ: ﺗﻘﺳﯾم ﺑﻧدی رﺷﺗﮫ ھﺎی ﺗﺣﺻﯾﻠﯽ آﻣوزش ﻋﺎﻟﯽ ﮐﮫ ﺷﺎﻣل ﭘﻧﺞ

## 3. 2-Step RAG with LCEL

In [32]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

#llm = init_chat_model("gpt-4o-mini", model_provider="openai", temperature=0)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Answer the question based only on the following context.
If you don't know the answer, say "I do not know. I do not have information about this."

Context:
{context}"""),
("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 2-Step RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test
response = rag_chain.invoke("When are applicants registered?")
#response = rag_chain.invoke("Who is this CV for?")
print(response)


I do not know. I do not have information about this.


In [33]:
# Test 2
response = rag_chain.invoke("According to which clause and regulation is the total score calculated?")
print(response)

The total score is calculated based on Clause 2 - 2 of the resolution of the Council of the Supreme Cultural Revolution, dated 15/04/1400, which states that the total educational score is determined by the weighted average of the scores obtained in general and specialized subjects.


In [34]:
# Test 2
response = rag_chain.invoke("According to what formula is the total score calculated?")
print(response)

The total score is calculated based on the weighted average of the scores obtained in general and specialized courses. Specifically, it is derived from the average of the weighted scores of the courses taken, as per the regulations set by the relevant educational authority.


In [35]:
# Test 2
response = rag_chain.invoke("Is chemistry more useful than mathematics?")
print(response)

I do not know. I do not have information about this.
